In [26]:
import pandas as pd
import os
import gcamreader

# Source

* Data Source:
    * December 2020 Motor Vehicle Registration Statistics, Ministry of Land, Infrastructure and Transport (MOLIT). (`../resources/VRS-2020-MOLIT`)

* Implemented Input Files
    * `/input/policy/korea-2035/transportation/ZEV_calibrate.xml`

We calibrate ZEV service output using Korea’s 2020 ZEV enrollment data. The number of registered vehicles is converted into performance units based on the assumptions specified in `basic_assumption.ipynb`. For passenger cars, calibration in the model is further disaggregated into the `Car` and `Large Car and Truck` categories. Detailed implementation steps are provided below.

In [12]:
df = pd.read_excel("../resources/VRS-2020-MOLIT.xlsx", sheet_name="10.연료별_등록현황", skiprows=[0, 1])
df

,연료별,시도별,Unnamed: 2,서울,부산,대구,인천,광주,대전,울산,...,경기,강원,충북,충남,전북,전남,경북,경남,제주,계
0,NaN,종별,용도별,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,휘발유,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,승용,비사업용,1611157.0,659191.0,578131.0,652762.0,302504.0,332114.0,286856.0,...,2895301.0,344805.0,380173.0,497637.0,383217.0,366207.0,628286.0,813623.0,163448.0,10982274.0
4,NaN,NaN,사업용,20149.0,25034.0,3160.0,145108.0,1352.0,1662.0,948.0,...,12163.0,1538.0,2461.0,2259.0,3586.0,52564.0,1287.0,30567.0,100882.0,404842.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
285,NaN,NaN,사업용,4931.0,8890.0,2033.0,5296.0,1826.0,1646.0,2171.0,...,11282.0,1723.0,2737.0,2978.0,2205.0,4874.0,5038.0,5478.0,645.0,63918.0
286,NaN,NaN,계,8930.0,10989.0,3641.0,7639.0,2990.0,2802.0,3001.0,...,21554.0,3652.0,4574.0,5455.0,4245.0,7616.0,8230.0,8722.0,1417.0,105937.0
287,NaN,소계,비사업용,2953615.0,1295675.0,1168056.0,1318740.0,659540.0,656976.0,554016.0,...,5771837.0,783899.0,829617.0,1110156.0,908511.0,950516.0,1426974.0,1671248.0,384368.0,22615643.0
288,NaN,NaN,사업용,203746.0,133365.0,51140.0,357702.0,32200.0,29453.0,21684.0,...,232289.0,25200.0,36003.0,39689.0,39636.0,149378.0,48415.0,116619.0,230974.0,1750336.0


In [13]:
df['연료별'] = df['연료별'].ffill()
df['시도별'] = df['시도별'].ffill()

In [14]:
df[(df['연료별'] == '전기') & (~df['계'].isna()) & (df['Unnamed: 2'] == '계')][['연료별', '시도별', 'Unnamed: 2',  '계']]

,연료별,시도별,Unnamed: 2,계
73,전기,승용,계,117616.0
76,전기,승합,계,1837.0
79,전기,화물,계,15436.0
82,전기,특수,계,73.0
85,전기,소계,계,134962.0


* 2020년 말 기준 전기승용차 등록대수(대): 117616.0
* 승용차 pass-km/veh/year = 33 * 365 * 1.26

In [15]:
# 전기승용 2020년 service output (million pass-km)
117616.0 * 33 * 365 * 1.26 / 1e6

1785.0227472000001

* 2020년 말 기준 전기버스 등록대수(대): 1837
* 버스 pass-km/veh/year = 824767

In [16]:
# 전기버스 2020년 service output (million pass-km)
1837 * 824767 / 1e6

1515.096979

* 2020년 말 기준 전기화물차 등록대수(대): 15436
* 화물차 ton-km/veh/year = 47 * 365 * 4.2

In [17]:
# 전기화물차 2020년 service output (million ton-km)
15436 * 47 * 365 * 4.2 / 1e6

1112.179236

In [18]:
df[(df['연료별'] == '수소') & (~df['계'].isna()) & (df['Unnamed: 2'] == '계')][['연료별', '시도별', 'Unnamed: 2',  '계']]

,연료별,시도별,Unnamed: 2,계
243,수소,승용,계,10831.0
246,수소,승합,계,75.0
249,수소,화물,계,0.0
252,수소,특수,계,0.0
255,수소,소계,계,10906.0


* 2020년 말 기준 수소승용차 등록대수(대): 10831
* 승용차 pass-km/veh/year = 33 * 365 * 1.26

In [19]:
# 수소승용 2020년 service output (million pass-km)
10831 * 33 * 365 * 1.26 / 1e6

164.3788377

* 2020년 말 기준 수소버스 등록대수(대): 75
* 버스 pass-km/veh/year = 824767

In [20]:
# 수소버스 2020년 service output (million pass-km)
75 * 824767 / 1e6

61.857525

* 2020년 말 기준 수소화물차 등록대수(대): 0
* 화물차 ton-km/veh/year = 47 * 365 * 4.2

In [21]:
# 전기화물차 2020년 service output (million ton-km)
0 * 47 * 365 * 4.2 / 1e6

0.0

In [29]:
dbpath = "../../output/"
dbfile = "database_basexdb"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', '..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Reference, Macro


In [30]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Reference', 'Macro']

In [31]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [32]:
i = 159
q = queries[i]
title = q.title
print(title)
df = conn.runQuery(q, scenarios=['Reference'], regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

transport service output by tech


,Units,scenario,region,sector,subsector,technology,Year,value
0,million pass-km,Reference,South Korea,trn_aviation_intl,International Aviation,BEV,2035,4.188620e-03
1,million pass-km,Reference,South Korea,trn_aviation_intl,International Aviation,BEV,2040,3.478500e-02
2,million pass-km,Reference,South Korea,trn_aviation_intl,International Aviation,BEV,2045,2.693970e-01
3,million pass-km,Reference,South Korea,trn_aviation_intl,International Aviation,BEV,2050,1.900340e+00
4,million pass-km,Reference,South Korea,trn_aviation_intl,International Aviation,BEV,2055,3.531650e+00
...,...,...,...,...,...,...,...,...
901,million ton-km,Reference,South Korea,trn_shipping_intl,International Ship,Liquids,2080,1.643780e+06
902,million ton-km,Reference,South Korea,trn_shipping_intl,International Ship,Liquids,2085,1.693100e+06
903,million ton-km,Reference,South Korea,trn_shipping_intl,International Ship,Liquids,2090,1.742860e+06
904,million ton-km,Reference,South Korea,trn_shipping_intl,International Ship,Liquids,2095,1.789120e+06


In [33]:
df[(df['Year'] == 2025) & (df['sector'].isin(['trn_pass_road_LDV_4W'])) & (df['technology'] == 'BEV')]

,Units,scenario,region,sector,subsector,technology,Year,value
405,million pass-km,Reference,South Korea,trn_pass_road_LDV_4W,Car,BEV,2025,12033.30
496,million pass-km,Reference,South Korea,trn_pass_road_LDV_4W,Large Car and Truck,BEV,2025,5741.57


In [34]:
car_share = 12033.30 / (12033.30+5741.57)
car_share

0.676983854171648

In [35]:
1785.02 * car_share, 1785.02 * (1-car_share) 

(1208.429719373475, 576.590280626525)

In [36]:
df[(df['Year'] == 2025) & (df['sector'].isin(['trn_pass_road_LDV_4W'])) & (df['technology'] == 'FCEV')]

,Units,scenario,region,sector,subsector,technology,Year,value
421,million pass-km,Reference,South Korea,trn_pass_road_LDV_4W,Car,FCEV,2025,434.735
512,million pass-km,Reference,South Korea,trn_pass_road_LDV_4W,Large Car and Truck,FCEV,2025,164.938


In [37]:
car_share = 434.735 / (434.735+164.938)
car_share

0.7249534329542935

In [38]:
164.38 * car_share, 164.38 * (1-car_share) 

(119.16784530902676, 45.212154690973236)